In [ ]:
import numpy as np

number_cities = 15 # количество городов
list_cities = np.random.random([number_cities, 2]) # спискок координат городов
print(list_cities)
print(list_cities[2, 1])
list_cities.shape

In [ ]:
import matplotlib.pyplot as plt

# нарисуем города
plt.scatter(list_cities[:, 0], list_cities[:, 1], c='red')

# пронумеруем города
for i in range(number_cities):
  plt.annotate(i, (list_cities[i][0], list_cities[i][1]))
plt.show()

In [ ]:
# нарисовать маршрут
def route_image(list_cities, route):
  # пронумеруем города
  for i in range(number_cities):
    plt.annotate(i, (list_cities[i][0], list_cities[i][1]), fontsize=14)

  # нарисуем города
  plt.scatter(list_cities[:, 0], list_cities[:, 1], c='red')

  # нарисуем маршрут
  x = [list_cities[route[-1]][0]]
  y = [list_cities[route[-1]][1]]
  for i in range(len(route)):
    x.append(list_cities[route[i]][0])
    y.append(list_cities[route[i]][1])
  plt.plot(x, y, c='blue')

  # выведем изображение
  plt.show()

In [ ]:
route = np.random.randint(number_cities, size=number_cities) # сгенерированный маршрут
print("Маршрут:", route)
route_image(list_cities, route)

In [ ]:
def route_length(list_cities, route):
  # штраф за не посещение города
  length_penalty = 0
  for i in range(len(list_cities)):
    if i not in route:
      length_penalty += 10

  # длина маршрута
  length = 0
  for i in range(len(route)-1):
    d = ((list_cities[route[i], 0]-list_cities[route[i+1], 0])**2 + (list_cities[route[i], 1]-list_cities[route[i+1], 1])**2)**0.5
    length = length + d


  return length_penalty + length

In [ ]:
print("Качество маршрута: ", route_length(list_cities, route))
print("Маршрут: ", route)
route_image(list_cities, route)

In [ ]:
number_routes = 1000 # количество маршрутов (особей)

# генерация маршрутов без повторений
routes = []
for i in range(number_routes):
  routes.append(np.random.permutation(number_cities))
routes = np.array(routes)

In [ ]:
print(routes)
routes[1]
len(routes)

In [ ]:
# качество особей (маршрутов)
def quality(list_cities, routes):
  # проведем оценку маршрутов
  length_penalty = np.zeros([len(routes)])

  for j in range(len(routes)):
    route = routes[j]
    length = 0
    for i in range(len(route)-1):
      d = ((list_cities[route[i], 0]-list_cities[route[i+1], 0])**2 + (list_cities[route[i], 1]-list_cities[route[i+1], 1])**2)**0.5
      length = length + d
    length_penalty[j] = length

  return length_penalty

In [ ]:
length_penalty = quality(list_cities, routes)

min_route = routes[np.argmin(length_penalty)]
print("Маршрут: ", min_route)
print("Качество маршрута:", route_length(list_cities, min_route))
route_image(list_cities, min_route)

In [ ]:
# мутация маршрута
def mutation(route):
  route1 = np.zeros([len(route)])
  g = len(route)
  for i in range(g):
    route1[i] = route[g-i-1]
  route = route1


  return route

In [ ]:
# скрещивание
def crossbreeding(route_1, route_2):

  route_pr = np.array((range(len(route_1))), float)

  route_0 = np.zeros([len(route_1)])

  route_3 = np.zeros([len(route_1)])

  for i in range(len(route_1)):
    if i <= 4:
      route_3[i] = route_1[i]
    if i > 4 and i <= (len(route_1)-4):
      route_3[i] = route_2[i]
    if i > (len(route_1)-4) and i <= (len(route_1)):
      route_3[i] = route_1[i]

  for i in range(len(route_1)):
    n = 0
    for j in range(len(route_1)):
      if route_pr[i] == route_3[j]:
        n = n + 1
    route_0[i] = n

  route_net = np.zeros([0])

  for i in range(len(route_1)):
    if route_0[i] == 0:
      route_net = np.append(route_net, route_pr[i])

  for i in range(len(route_1)):
    if route_0[i] > 1:
      povtor = route_pr[i]
      n = 0
      for j in range(len(route_1)):
        if n >= 1 and route_3[j] == povtor:
          route_3[j] = len(route_1) + 1
        if route_3[j] == povtor:
          n = n+1

  for i in range(len(route_net)):
    net = route_net[i]
    for j in range(len(route_1)):
      if route_3[j] == len(route_1) + 1:
        route_3[j] = net
        break


  return route_3

In [ ]:
from copy import deepcopy as dcopy

number_iterations = 20000 # количество итераций улучшения

# 2. Оценка качества особей. Функция приспособленности.
# length_penalty - список оценок всех особей популяции
length_penalty = quality(list_cities, routes)

for i in range(number_iterations):
  # отсортируем особи в порядке от наиболее приспособленных к наименее приспособленным
  sort_indx = length_penalty.argsort() # индексы отсортированных элементов
  length_penalty = length_penalty[sort_indx] # меняем оценки качества особей местами в соответствии с отсортированными индексами
  routes = routes[sort_indx] # меняем особи местами в соответствии с отсортированными индексами

  # выбираем особи для операций мутации и скрещивания
  route_index_1 = np.random.randint(number_routes)
  route_index_2 = np.random.randint(number_routes)

  # 3. Отбор наиболее сильных решений. Селекция
  # (упрощенный вариант селекции с заменой одной наименее приспособленной особи на новую)
  if np.random.random()<0.5:
    # 4.1 мутация
    #routes[-1] = mutation(routes[route_index_1])
    pass
  else:
    # 4.2 скрещивание
    routes[-1] = crossbreeding(routes[route_index_1], routes[route_index_2])

  # 2. Оценка качества особей. Функция приспособленности.
  length_penalty = quality(list_cities, routes)

  # вывод промежуточного результата
  if i%(number_iterations//5) == 0:
    print("Итерация:", i)
    print("Маршрут: ", routes[0])
    print("Качество маршрута:", route_length(list_cities, routes[0]))
    route_image(list_cities, routes[0])
    print('\n\n')

In [ ]:
length_penalty = quality(list_cities, routes)

min_route = routes[np.argmin(length_penalty)]
print("Маршрут: ", min_route)
print("Качество маршрута:", route_length(list_cities, min_route))
route_image(list_cities, min_route)